In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import pprint

In [3]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

### Defining the LLM

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

### Defining the Graph state

State is the object that is passed between nodes in the graph.

In [5]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AnyMessage # type: ignore
import operator
from IPython.display import Image, display

In [6]:
class AgentState(TypedDict):
    query: str
    agent_outcome: List[AnyMessage]
    chat_history: Annotated[list, operator.add]

By default, `StateGraph` operates with a single schema, and all nodes(agents) are expected to communicate using that schema. However, it's also possible to define distinct `input` and `output` schemas for a graph.

When distinct schemas are specified, an internal schema will still be used for communication between nodes. The `input` schema ensures that the provided input matches the expected structure, while the `output` schema filters the internal data to return only the relevant information according to the defined output schema.

In [7]:
class UserInput(TypedDict):
    query: str

In [8]:
class FinalOutput(TypedDict):
    answer: str

### Define the agent (node)

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def research_agent(data:AgentState) -> FinalOutput:
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI assistant"
                "\nUser Query: {query}"
            ),
            MessagesPlaceholder(variable_name="chat_history"),
        ]
    )
    agent = prompt | llm
    result = agent.invoke(data)
    data['agent_outcome'] = [result]
    return {
                "answer": result.content,
                "chat_history": [result]
            }

### Defining the workflow (graph)

In [ ]:
from langgraph.graph import END, StateGraph

## Initialising the workflow
workflow = StateGraph(AgentState, input=UserInput, output=FinalOutput)

## Adding node (agent) to the graph (workflow)
workflow.add_node("research", research_agent)

## Setting the entry point of the graph
workflow.set_entry_point("research")

## Compiling the graph
app = workflow.compile()
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
inputs = {
    "query": "What are Small Language Models?",
}

app.invoke(input=inputs)

We can see the output schema `FinalOutput` constrains the output to only the `answer` key.